In [1]:
import pandas as pd
import numpy as np
from PIL import Image
import io
import torch
from tqdm.auto import tqdm
from qdrant_client import QdrantClient
from qdrant_client.http import models
from models.scold.model import LVL
from transformers import RobertaTokenizer
from torchvision import transforms
import glob
import pyarrow.dataset as ds
import dask.dataframe as dd
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def sample_by_caption_efficient(data_path, n_samples=5, random_state=42):
    
    # Load all parquet files
    df = dd.read_parquet(f"{data_path}/*.parquet")
    
    print(f"Total rows: {len(df)}")
    print(f"Sampling {n_samples} examples from each caption...")
    
    def sample_partition(partition_df):
        sampled_groups = []
        for caption, group in partition_df.groupby('caption'):
            sample_size = min(n_samples, len(group))
            if sample_size > 0:
                sampled = group.sample(n=sample_size, random_state=random_state)
                sampled_groups.append(sampled)
        
        if sampled_groups:
            return pd.concat(sampled_groups, ignore_index=True)
        else:
            return pd.DataFrame()
    
    sampled_partitions = df.map_partitions(sample_partition, meta=df._meta)
    
    def final_sample(group_df):
        if len(group_df) > n_samples:
            return group_df.sample(n=n_samples, random_state=random_state)
        return group_df
    
    result = sampled_partitions.groupby('caption').apply(final_sample, meta=df._meta)
    result = result.reset_index(drop=True)
    
    return result

data_path = "data/leafnet"
n_samples_per_caption = 5 

sampled_df = sample_by_caption_efficient(data_path, n_samples=n_samples_per_caption)

print(f"\nTotal sampled rows: {len(sampled_df)}")

caption_counts = sampled_df['caption'].value_counts().compute()
print(f"\nSample distribution:")
print(caption_counts.sort_index())

sampled_df_pd = sampled_df.compute()


Total rows: 121337
Sampling 5 examples from each caption...


d:\Workspace\Repository\skripsi\.skripsi-venv\Lib\site-packages\dask\dataframe\groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
d:\Workspace\Repository\skripsi\.skripsi-venv\Lib\site-packages\dask\dataframe\groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
d:\Workspace\Repository\skripsi\.skripsi-venv\Lib\site-package


Total sampled rows: 445


d:\Workspace\Repository\skripsi\.skripsi-venv\Lib\site-packages\dask\dataframe\groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
d:\Workspace\Repository\skripsi\.skripsi-venv\Lib\site-packages\dask\dataframe\groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
d:\Workspace\Repository\skripsi\.skripsi-venv\Lib\site-package


Sample distribution:
caption
a image of Apple healthy leaves with leaves appearing normal and healthy                                                                                                                                               5
a image of Apple leaves diseased by Black rot with symptoms of small black spots or concentric rings, leading to shriveling or drying.                                                                                 5
a image of Apple leaves diseased by Cedar-Apple Rust with symptoms of yellow to bright orange-red spots, black dots, and finger-like fungal tubes on the leaf surface.                                                 5
a image of Apple leaves diseased by Scab with symptoms of olive-green to dark brown spots on leaves, leading to yellowing                                                                                              5
a image of Black Pepper healthy leaves with leaves appearing normal and healthy                       

d:\Workspace\Repository\skripsi\.skripsi-venv\Lib\site-packages\dask\dataframe\groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
d:\Workspace\Repository\skripsi\.skripsi-venv\Lib\site-packages\dask\dataframe\groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
d:\Workspace\Repository\skripsi\.skripsi-venv\Lib\site-package

In [3]:
sampled_df_pd['caption'].nunique()

89

In [4]:
# import re

# # Step 1: Get unique captions
# unique_captions = sampled_df_pd['caption'].unique()

# # Step 2: Build mapping
# label_mapping = {}
# for i, cap in enumerate(unique_captions):
#     # Try to extract crop and disease/condition
#     match = re.search(r"of (.*?) leaves diseased by (.*?) with", cap)
#     match_healthy = re.search(r"of (.*?) healthy leaves", cap)
    
#     if match:
#         crop = match.group(1).replace(" ", "")
#         disease = match.group(2).replace(" ", "").replace("-", "").replace(" ", "")
#         label = f"{crop}_{disease}"
#     elif match_healthy:
#         crop = match_healthy.group(1).replace(" ", "")
#         label = f"{crop}_Healthy"
#     else:
#         # Fallback: generic label with index
#         label = f"Label_{i}"
    
#     label_mapping[cap] = label

# # Step 3: Add labels to dataframe
# sampled_df_pd['label'] = sampled_df_pd['caption'].map(label_mapping)

In [5]:
print([hex(ord(c)) for c in "Two-spotted spider mites"])
# vs
print([hex(ord(c)) for c in "Two‑spotted spider mites"])


['0x54', '0x77', '0x6f', '0x2d', '0x73', '0x70', '0x6f', '0x74', '0x74', '0x65', '0x64', '0x20', '0x73', '0x70', '0x69', '0x64', '0x65', '0x72', '0x20', '0x6d', '0x69', '0x74', '0x65', '0x73']
['0x54', '0x77', '0x6f', '0x2011', '0x73', '0x70', '0x6f', '0x74', '0x74', '0x65', '0x64', '0x20', '0x73', '0x70', '0x69', '0x64', '0x65', '0x72', '0x20', '0x6d', '0x69', '0x74', '0x65', '0x73']


In [6]:
import json

# If your JSON is saved in a file, e.g. "labels.json"
with open("data/leafnet/labels.json", "r") as f:
    caption_to_label = json.load(f)

sampled_df_pd['caption'] = sampled_df_pd['caption'].str.replace('‐', '-')

# Map captions to new column
sampled_df_pd["label"] = sampled_df_pd["caption"].map(caption_to_label)

sampled_df_pd['label'].nunique()


88

In [7]:
sampled_df_pd['label'].unique()

array(['BlackPepper_Healthy', 'BlackPepper_YellowMottleVirus',
       'Cucumber_DownyMildew', 'Maize_CercosporaGrayLeafSpot',
       'Cashew_LeafMiner', 'Tomato_EarlyBlight', 'Cashew_RedRust',
       'Tea_RedLeafSpot', 'Tomato_LeafMold', 'Cucumber_GummyStemBlight',
       'Mango_SootyMould', 'Peach_BacterialSpot', 'Cherry_PowderyMildew',
       'Tomato_LateBlight', 'Cassava_BacterialBlight',
       'Strawberry_LeafScorch', 'Sugarcane_BandedChlorosis',
       'Tea_LeafBlight', 'Grape_PowderyMildew', 'BlackPepper_LeafBlight',
       'Cashew_Anthracnose', 'Cucumber_Anthracnose', 'Coffee_Cercospora',
       'Strawberry_Healthy', 'Coffee_Phoma', 'Maize_NorthernLeafBlight',
       'Sugarcane_Smut', 'Pepper_Healthy', 'Potato_EarlyBlight',
       'Squash_PowderyMildew', 'Maize_MaizeLethalNecrosis',
       'Sugarcane_BrownRust', 'Cassava_MosaicVirus', 'Cassava_GreenMite',
       'Cucumber_BacterialWilt', 'Tomato_VerticilliumWilt',
       'Coffee_Miner', 'Grape_BlackRot', 'Mango_Anthracnose',
  

In [ ]:
sampled_df_pd.to_parquet('data/leafnet/leafnet_sampled.parquet', index=False)